# Almond Tree Segmentation Pipeline (Improved)

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm import tqdm
import glob
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


# Device setup
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Custom COCO Dataset
class COCOSegmentationDataset(Dataset):
    def __init__(self, img_dir, ann_path, transform=None):
        self.img_dir = img_dir
        self.coco = COCO(ann_path)
        self.image_ids = list(self.coco.imgs.keys())
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.img_dir, img_info['file_name'])
    
        # Load image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
        # Load annotations and build binary mask
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
    
        for ann in anns:
            mask = np.maximum(mask, self.coco.annToMask(ann))
    
        # ✅ Only now: check if resizing needed (after mask is defined)
        if mask.shape[:2] != image.shape[:2]:
            mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
    
        # Apply transforms
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = (augmented['mask'] > 0).unsqueeze(0).float()  # Ensure shape [1, H, W]
    
        return image, mask


### Transformations

In [ ]:
# Constants
IMAGE_SIZE = 1024
BATCH_SIZE = 4

# Safer, more controlled train augmentations
train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Equalize(mode='cv', p=0.5),
    A.CLAHE(clip_limit=3.0, p=0.2),
    A.Emboss(alpha=(0.2, 0.5), strength=(0.2, 0.6), p=0.2),     # deepens lines like branches
    A.RandomGamma(gamma_limit=(60, 140), p=0.5),
    A.Normalize(),
    ToTensorV2()
])

# Clean validation transform
val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(),
    ToTensorV2()
])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import glob
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch

# Constants
IMAGE_SIZE = 1024

# Load image paths
image_paths = sorted(glob.glob("/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-6/train/*.jpg"))[:5]

# Transform: original (just resized)
original_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    ToTensorV2()
])

strong_aug_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Equalize(mode='cv', p=0.5),
    A.CLAHE(clip_limit=3.0, p=0.2),
    A.Emboss(alpha=(0.2, 0.5), strength=(0.2, 0.6), p=0.2),     # deepens lines like branches
    A.RandomGamma(gamma_limit=(60, 140), p=0.5),
    A.Normalize(),
    ToTensorV2()
])


# Denormalization for display
def denormalize_image(tensor_img, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    np_img = tensor_img.permute(1, 2, 0).cpu().numpy()
    np_img = np_img * np.array(std) + np.array(mean)
    np_img = np.clip(np_img, 0, 1)
    return np_img

# Display function
def show_strong_augmentations(image_paths):
    for img_path in image_paths:
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Apply transforms
        original = original_transform(image=image)["image"]
        augmented_tensor = strong_aug_transform(image=image)["image"]
        augmented = denormalize_image(augmented_tensor)

        # Plot side-by-side
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(original.permute(1, 2, 0).cpu().numpy())
        plt.title("Original (Resized Only)")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(augmented)
        plt.title("Safe Strong Augmentation")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

# Run
show_strong_augmentations(image_paths)


### Paths Datasets and Dataloaders

In [ ]:
# Paths
train_img_dir = "/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-6/train"
train_ann_path = os.path.join(train_img_dir, "_annotations.coco.json")
val_img_dir = "/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-6/valid"
val_ann_path = os.path.join(val_img_dir, "_annotations.coco.json")

# Datasets and Dataloaders
train_dataset = COCOSegmentationDataset(train_img_dir, train_ann_path, train_transform)
val_dataset = COCOSegmentationDataset(val_img_dir, val_ann_path, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

### Model

In [ ]:
# Updated Model: Use EfficientNet-B3 with no final activation
model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b3",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(device)

In [ ]:
import torch
import torch.nn as nn

# Manual implementation of Focal Tversky Loss
class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=0.75, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)  # Apply sigmoid here since activation=None
        targets = targets

        TP = (inputs * targets).sum(dim=(1, 2, 3))
        FP = ((1 - targets) * inputs).sum(dim=(1, 2, 3))
        FN = (targets * (1 - inputs)).sum(dim=(1, 2, 3))

        tversky = (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)
        focal_tversky = (1 - tversky) ** self.gamma

        return focal_tversky.mean()


In [ ]:
loss_fn = FocalTverskyLoss(alpha=0.3, beta=0.7, gamma=0.75)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Scheduler with fixed closing parenthesis
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2)

In [ ]:
def dice_coef(preds, targets, threshold=0.5, eps=1e-6):
    preds = (torch.sigmoid(preds) > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    dice = (2. * intersection + eps) / (union + eps)
    return dice.mean()

def iou_score(preds, targets, threshold=0.5, eps=1e-6):
    preds = (torch.sigmoid(preds) > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean()


In [ ]:
import matplotlib.pyplot as plt

def plot_learning_curves(history, metrics=None):
    if metrics is None:
        metrics = ['train_loss', 'val_loss', 'train_dice', 'val_dice', 'train_iou', 'val_iou']
    
    plt.figure(figsize=(15, 10))
    for metric in metrics:
        plt.plot(history[metric], label=metric)

    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Training and Validation Curves")
    plt.legend()
    plt.grid(True)
    plt.show()

### Training and Evaluation

In [ ]:
# Initialize history tracking
history = {
    'train_loss': [],
    'val_loss': [],
    'train_dice': [],
    'val_dice': [],
    'train_iou': [],
    'val_iou': []
}

# Best score tracking
best_val_dice = 0
patience = 5
epochs_no_improve = 0

# Scheduler for dynamic learning rate
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

# Main training loop
for epoch in range(1, 31):
    model.train()
    train_loss = 0
    train_dice = 0
    train_iou = 0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_dice += dice_coef(outputs, masks).item()
        train_iou += iou_score(outputs, masks).item()

    model.eval()
    val_loss = 0
    val_dice = 0
    val_iou = 0

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            val_loss += loss_fn(outputs, masks).item()
            val_dice += dice_coef(outputs, masks).item()
            val_iou += iou_score(outputs, masks).item()

    # Averages
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    avg_train_dice = train_dice / len(train_loader)
    avg_val_dice = val_dice / len(val_loader)
    avg_train_iou = train_iou / len(train_loader)
    avg_val_iou = val_iou / len(val_loader)

    # Scheduler step
    scheduler.step(avg_val_loss)

    # Logging
    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
          f"Train Dice: {avg_train_dice:.4f} | Val Dice: {avg_val_dice:.4f} | "
          f"Train IoU: {avg_train_iou:.4f} | Val IoU: {avg_val_iou:.4f}")

    # Check for improvement
    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        epochs_no_improve = 0

        # ✅ Full checkpoint saving
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': avg_val_dice,
            'val_loss': avg_val_loss
        }, "best_model.pth")

        print("✅ Saved new best model with optimizer and metrics")
    else:
        epochs_no_improve += 1
        print(f"⏳ No improvement for {epochs_no_improve} epochs")

    # ⛔ Early stopping
    if epochs_no_improve >= patience:
        print("⛔ Early stopping triggered")
        break

    # Update history
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_dice'].append(avg_train_dice)
    history['val_dice'].append(avg_val_dice)
    history['train_iou'].append(avg_train_iou)
    history['val_iou'].append(avg_val_iou)


In [ ]:
plot_learning_curves(history)


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# 1. Define test transform
test_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE), 
    A.Normalize(),  # Uses ImageNet stats
    ToTensorV2(),
])

# 2. Load test dataset
test_dataset = COCOSegmentationDataset(
    img_dir="/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-3/test/",  
    ann_path="/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-3/test/_annotations.coco.json", 
    transform=test_transform
)

# 3. Recreate the model with the same architecture used during training
model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b3",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(device)

# 4. Load the full checkpoint and restore the model weights
checkpoint = torch.load("best_model.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# 5. Denormalize function for visualization
def denormalize_image(tensor_img, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    if isinstance(tensor_img, torch.Tensor):
        tensor_img = tensor_img.detach().cpu()
    np_img = tensor_img.permute(1, 2, 0).numpy()
    np_img = np_img * np.array(std) + np.array(mean)
    np_img = np.clip(np_img, 0, 1)
    np_img = (np_img * 255).astype(np.uint8)
    return np_img

# 6. Prediction + Visualization
def visualize_predictions(model, dataset, device, max_samples=None):
    n_samples = len(dataset) if max_samples is None else min(len(dataset), max_samples)
    for i in tqdm(range(n_samples), desc="Predicting"):
        image, true_mask = dataset[i]
        image_tensor = image.unsqueeze(0).to(device)

        with torch.no_grad():
            pred_mask = model(image_tensor)

        pred_mask = (pred_mask.squeeze().cpu().numpy() > 0.5).astype(np.uint8)
        image_np = denormalize_image(image)
        true_mask_np = true_mask.squeeze().cpu().numpy()

        # Plot
        fig, axs = plt.subplots(1, 3, figsize=(12, 4))
        axs[0].imshow(image_np)
        axs[0].set_title("Original Image")
        axs[1].imshow(true_mask_np, cmap='gray')
        axs[1].set_title("Ground Truth")
        axs[2].imshow(pred_mask, cmap='gray')
        axs[2].set_title("Predicted Mask")
        for ax in axs:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

# 7. Run predictions
visualize_predictions(model, test_dataset, device, max_samples=10)
